# Microwave Fixed-Frequency / Gain Calibration

Test microwave output at a **single fixed frequency** instead of sweeping. Useful for:
- Calibrating microwave power vs gain (e.g. with a power meter)
- Testing ODMR contrast at one frequency for different gains

- **Section 0** – Setup & connection
- **Section 1** – Configuration (set frequency and gain)
- **Section 2** – Run at fixed frequency (one measurement)
- **Section 3** – Optional: gain sweep (PL vs gain)

Run `01_basic_nv_testing.ipynb` first to set up connection and config, or run Section 0 here.

---
## Section 0: Setup & Connection

In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from copy import copy
import qickdawg as qd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# ===== EDIT THIS: Set your RFSoC IP address =====
RFSOC_IP = '192.168.0.101'

qd.start_client(RFSOC_IP)
print(f"Connected to RFSoC at {RFSOC_IP}")

Connected to RFSoC at 192.168.0.101


---
## Section 1: Configuration — Fixed Frequency & Gain

Set the **microwave frequency** (MHz) and **gain** (0–32767). The microwave will run at this single frequency — no sweep.

In [8]:
config = qd.NVConfiguration()

# ADC & laser (match your 01_basic_nv_testing setup)
config.adc_channel = 0
config.mw_channel = 1
config.mw_nqz = 1
config.laser_gate_pmod = 0
config.relax_delay_tns = 500

# ===== EDIT: Fixed microwave frequency (no sweep) =====
config.mw_fMHz = 2875

# ===== EDIT: Microwave gain =====
config.mw_gain = 5000

# ODMR-like timing (no add_linear_sweep — FixedFrequencyODMR uses mw_fMHz directly)
config.readout_integration_treg = 2**16 - 1
config.pre_init = False
config.reps = 100

print(f"Configuration:")
print(f"  Microwave: {config.mw_fMHz} MHz (fixed, no sweep)")
print(f"  Gain:     {config.mw_gain}")
print(f"  Reps:     {config.reps}")

Configuration:
  Microwave: 2874.9999996185306 MHz (fixed, no sweep)
  Gain:     5000
  Reps:     100


---
## Section 2: Run at Fixed Frequency

Runs one ODMR-style measurement (MW on / MW off) at the configured frequency and gain. Change `config.mw_gain` in Section 1 and re-run both cells to test different gains.

In [9]:
prog = qd.FixedFrequencyODMR(config)
d = prog.acquire(progress=True)

print(f"\nFixed frequency: {d.frequencies[0]:.2f} MHz")
print(f"Signal (MW on):    {d.signal[0]:.4f} ADC units")
print(f"Reference (MW off): {d.reference[0]:.4f} ADC units")
print(f"Contrast:          {d.contrast[0]:.4f} ADC units")
print(f"Contrast %:        {d.contrast_percent[0]:.2f}%")

  0%|          | 0/100 [00:00<?, ?it/s]


Fixed frequency: 2875.00 MHz
Signal (MW on):    7.6076 ADC units
Reference (MW off): 6.9023 ADC units
Contrast:          0.7054 ADC units
Contrast %:        10.22%


---
## Section 3: Optional — Gain Sweep (PL vs Gain)

Sweep over different gain values at the same frequency. Plot PL signal and contrast vs gain to calibrate how much microwave power you get for each gain.

In [ ]:
# Gain range to sweep (adjust as needed)
GAIN_MIN = 1000
GAIN_MAX = 15000
GAIN_STEP = 1000

gains = np.arange(GAIN_MIN, GAIN_MAX + 1, GAIN_STEP, dtype=int)
print(f"Sweeping {len(gains)} gains: {gains[0]} .. {gains[-1]}")

In [ ]:
signal_arr = []
reference_arr = []
contrast_arr = []

for i, g in enumerate(gains):
    cfg = copy(config)
    cfg.mw_gain = int(g)
    prog = qd.FixedFrequencyODMR(cfg)
    d = prog.acquire(progress=False)
    signal_arr.append(d.signal[0])
    reference_arr.append(d.reference[0])
    contrast_arr.append(d.contrast[0])
    print(f"  gain {g}: signal={d.signal[0]:.4f}, contrast={d.contrast[0]:.4f}")

signal_arr = np.array(signal_arr)
reference_arr = np.array(reference_arr)
contrast_arr = np.array(contrast_arr)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 6))
ax1.plot(gains, signal_arr, 'b-o', label='Signal (MW on)', markersize=4)
ax1.plot(gains, reference_arr, 'r-s', label='Reference (MW off)', markersize=4)
ax1.set_ylabel('PL (ADC units)')
ax1.legend()
ax1.grid(True)

ax2.plot(gains, contrast_arr, 'g-o', markersize=4)
ax2.set_xlabel('Microwave gain')
ax2.set_ylabel('Contrast (ADC units)')
ax2.set_title(f'Contrast vs gain at {config.mw_fMHz} MHz (fixed)')
ax2.grid(True)

plt.tight_layout()
plt.show()